# Extended Data Figure 7 — Leave-one-out drug target enrichment

Sensitivity analysis for drug-target enrichment:

**(a)** Leave-one-out by **therapeutic area (TA)** — OR (95 % CI) for approved targets
after excluding each TA from the enrichment calculation.  
**(b)** Leave-one-out by **target class** — OR (95 % CI) after restricting ChEMBL
to targets in each L1 class.

Numbers in parentheses show the remaining approved T-I pairs after exclusion.

**Source notebook:** `chapters/02-analysis/06-target-enrichment/07-split_by_TA_target_class.ipynb`  
**Data:** `data/25.06/output/`, `data/intermediate_files/l2g_full_for_enrichment/`


## Setup


In [ ]:
from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.study_locus import StudyLocus
from gentropy.method.drug_enrichment_from_evid import chemblDrugEnrichment
from pyspark.sql import functions as f

In [ ]:
session = Session(extended_spark_conf={"spark.driver.memory": "40G"})

## Paths


In [ ]:
from manuscript_methods import paper

path_to_release_folder = str(paper.ROOT / "data" / "25.06") + "/"
path_to_intermediate_data_folder = str(paper.DERIVED) + "/"
figure_dir = str(paper.ROOT / "chapters" / "05-figures-supplementary" / "extended_data")

l2g_enrichment_path = path_to_intermediate_data_folder + "prioritised_genes_diseases"
chembl_evidence_path = path_to_release_folder + "output/evidence/sourceId=chembl"
disease_index_path = path_to_release_folder + "output/disease/disease.parquet"
target_index_path = path_to_release_folder + "output/target"

## Load data


In [ ]:
sl = StudyLocus.from_parquet(session, path_to_release_folder + "output/credible_set")
si = StudyIndex.from_parquet(session, path_to_release_folder + "output/study")

disease_index = session.spark.read.parquet(disease_index_path)
chembl_evidence = session.spark.read.parquet(chembl_evidence_path)
target = session.spark.read.parquet(target_index_path)

l2g_full = session.spark.read.parquet(l2g_enrichment_path)
print(f"L2G for enrichment: {l2g_full.count():,}")

## Panel (a) — Leave-one-out by therapeutic area

For each TA, re-run the enrichment analysis excluding that TA from the disease list and report the OR for approved targets.


In [ ]:
import pandas as pd

# List therapeutic areas to analyse (exclude meta-categories)
unique_tas = (
    disease_index.select(f.explode(f.col("therapeuticAreas")).alias("ta")).distinct().rdd.map(lambda x: x[0]).collect()
)
ta_to_exclude = {"EFO_0001444", "MONDO_0045024", "GO_0008150", "EFO_0000651"}
ta_to_analyse = [ta for ta in unique_tas if ta not in ta_to_exclude]
print(f"Therapeutic areas to analyse: {len(ta_to_analyse)}")

In [ ]:
# Build evidence table once — reused across the leave-one-out loop
evidence = chemblDrugEnrichment.to_disease_target_evidence(
    table_with_score=l2g_full.drop("diseaseIds"),
    score_column="score",
    datasource_id="l2g",
    study_locus=sl,
    study_index=si,
    min_score=0.1,
)

# Run leave-one-out enrichment for each TA
all_enrich_ta = []
for ta in ta_to_analyse:
    enrich = chemblDrugEnrichment.drug_enrichemnt_from_evidence(
        evid=evidence,
        disease_index_orig=disease_index,
        chembl_orig=chembl_evidence,
        indirect_assoc_score_thr=0.1,
        efo_ancestors_to_remove=["MONDO_0045024", ta],
    )
    enrich["excluded_ta"] = ta
    all_enrich_ta.append(enrich)

df_ta = pd.concat(all_enrich_ta, ignore_index=True)
# Keep approved (phase 4+) results
df_ta_phase4 = df_ta[df_ta["clinicalPhase"] == "4+"].copy()

# Attach human-readable TA names
ta_names = disease_index.filter(f.col("id").isin(unique_tas)).select("id", "name").toPandas()
df_ta_phase4 = df_ta_phase4.merge(ta_names, left_on="excluded_ta", right_on="id", how="left")
df_ta_phase4 = df_ta_phase4.sort_values("odds_ratio")
print(df_ta_phase4[["name", "odds_ratio", "ci_low", "ci_high", "total_indirect_assoc"]].head())

## Panel (b) — Leave-one-out by target class


In [ ]:
# Extract L1 target classes
l1_classes = (
    target.select(f.explode(f.col("targetClass")).alias("tc"))
    .filter(f.col("tc.level").contains("l1"))
    .select("tc.label")
    .distinct()
    .orderBy("label")
    .rdd.map(lambda x: x[0])
    .collect()
)
print(f"L1 target classes: {l1_classes}")

In [ ]:
all_enrich_tc = []
for tid in l1_classes:
    # Leave-one-OUT: exclude targets belonging to this class from ChEMBL
    target_subset = (
        target.filter(f.array_contains(f.col("targetClass.label"), tid))
        .select("id")
        .withColumnRenamed("id", "targetId")
    )
    chembl_filtered = chembl_evidence.join(target_subset, on="targetId", how="left_anti")

    enrich = chemblDrugEnrichment.drug_enrichemnt_from_evidence(
        evid=evidence,
        disease_index_orig=disease_index,
        chembl_orig=chembl_filtered,
        indirect_assoc_score_thr=0.1,
        efo_ancestors_to_remove=["MONDO_0045024"],
    )
    enrich["target_class"] = tid
    all_enrich_tc.append(enrich)

df_tc = pd.concat(all_enrich_tc, ignore_index=True)
df_tc_phase4 = df_tc[df_tc["clinicalPhase"] == "4+"].copy()
df_tc_phase4 = df_tc_phase4.sort_values("odds_ratio")
print(
    df_tc_phase4[
        ["target_class", "odds_ratio", "ci_low", "ci_high", "total_indirect_assoc", "yes_evid-high_clinphase"]
    ].head()
)

## Extended Data Figure 7 — Forest plots


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

n_ta = len(df_ta_phase4)
n_tc = len(df_tc_phase4)
fig, (ax_a, ax_b) = plt.subplots(
    2,
    1,
    figsize=(7, n_ta * 0.35 + n_tc * 0.35),
    gridspec_kw={"hspace": 0.15},
)


def forest_plot(ax, df, y_col, panel_label, x_label="OR (95% CI)"):
    df = df.reset_index(drop=True)
    y_pos = np.arange(len(df))
    ax.errorbar(
        df["odds_ratio"],
        y_pos,
        xerr=[df["odds_ratio"] - df["ci_low"], df["ci_high"] - df["odds_ratio"]],
        fmt="o",
        color="steelblue",
        ecolor="steelblue",
        capsize=3,
        markersize=5,
        alpha=0.85,
    )
    ax.axvline(x=1, linestyle="--", color="black", linewidth=0.8)
    ax.set_xlim(2.5, 5)
    ax.set_yticks(y_pos)
    row_labels = [
        f"{row[y_col]}  ({int(row['total_indirect_assoc'])} / {int(row['yes_evid-high_clinphase'])})"
        for _, row in df.iterrows()
    ]
    ax.set_yticklabels(row_labels, fontsize=8)
    ax.set_xlabel(x_label, fontsize=9)
    ax.set_title(panel_label, fontsize=10, fontweight="bold", loc="left")
    ax.grid(axis="x", linestyle="--", alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)


forest_plot(
    ax_a,
    df_ta_phase4.sort_values("name"),
    y_col="name",
    panel_label="(a) Leave-one-out by therapeutic area",
)
forest_plot(
    ax_b,
    df_tc_phase4.sort_values("target_class"),
    y_col="target_class",
    panel_label="(b) Leave-one-out by target class",
)

fig.savefig(f"{figure_dir}/extended_figure_7.pdf", dpi=300, bbox_inches="tight")
plt.show()